# Baselines on the Seven Cutoffs

Every numerical baseline, scored with CRPS on the seven frozen forecast origins
(4 event, 3 quiet -- see [`CUTOFFS.md`](CUTOFFS.md)) at horizons 1/2/4/8/13 weeks:
**35 scored points per model**.

The predictors come from `aieng.forecasting.methods` where one exists, plus three
local ones written for reasons documented in their modules:

| Predictor | Source | Why it is in the lineup |
|---|---|---|
| `last_value_naive` | package | the floor: zero-uncertainty random walk |
| `darts_ets` | package | random walk + honest bands (fits alpha = 1) |
| `darts_autoarima` | package | AICc-selected ARIMA, picks d=1 everywhere |
| `darts_kalman` | package | kept to show the defect below -- do not cite alone |
| `kalman_fixed` | [`kalman_fixed.py`](kalman_fixed.py) | same model with the multi-step variance bug corrected |
| `darts_lightgbm` | package | kept to show what level-features do to a tree |
| `lgbm_diff` | [`lgbm_differenced.py`](lgbm_differenced.py) | the same booster on changes, which is the fair test |
| `darts_linreg` | package | linear control; its coefficients sum to ~1 |
| `prophet_weekly` | [`prophet_baseline.py`](prophet_baseline.py) | trend + yearly seasonality hypothesis |
| `seasonal_naive_52` | [`seasonal_naive.py`](seasonal_naive.py) | pure annual-cycle control |

**The finding, up front:** every model that beats the naive does so by wrapping the
*same* point forecast (carry the last price forward) in calibrated uncertainty.
ETS drives its smoothing parameter to the boundary (alpha = 1), AutoARIMA picks d=1
at every origin, N4SID identifies |eig| = 0.999, and linear regression's lag
coefficients sum to ~1 -- four independent estimators concluding *random walk*.
The two models that hypothesise calendar structure instead (Prophet, seasonal
naive) lose to the floor by ~2x. So the bar the news-reading agent has to clear
is **calibration and event anticipation, not point accuracy**.

**Data governance:** MPOB is approved for use (2026-08-11), locally and in Coder;
the raw series must never be committed. `data/mpob/` stays gitignored -- populate
it with `uv run python scripts/fetch_mpob.py`.


---
## 1. Setup

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import pandas as pd


ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "implementations"))
warnings.filterwarnings("ignore")

from aieng.forecasting.methods import (
    DartsAutoARIMAPredictor,
    DartsExponentialSmoothingPredictor,
    DartsKalmanForecasterPredictor,
    DartsLightGBMPredictor,
    DartsLinearRegressionPredictor,
    LastValuePredictor,
)
from cpo.baselines import (
    DEFAULT_NUM_SAMPLES,
    attach_actuals,
    coverage,
    load_spec,
    mc_noise,
    mpob_service,
    per_origin,
    predictions_frame,
    run_predictor,
    skill_scores,
    summarise,
)
from cpo.data import MPOB_WEEKLY_SERIES_ID, naive_utc_now
from cpo.kalman_fixed import FixedKalmanPredictor
from cpo.lgbm_differenced import DifferencedLightGBMPredictor
from cpo.plots import plot_crps_by_horizon, plot_fan_grid, plot_predictor_comparison
from cpo.prophet_baseline import WeeklyProphetPredictor
from cpo.seasonal_naive import SeasonalNaivePredictor


spec = load_spec()
svc = mpob_service()
print(f"origins  : {[f'{d:%Y-%m-%d}' for d in spec.origins()]}")
print(
    f"horizons : {spec.task.horizons} weeks -> {len(spec.origins()) * len(spec.task.horizons)} scored points per model"
)

origins  : ['2024-02-02', '2024-05-03', '2024-08-30', '2024-11-29', '2025-02-28', '2025-06-20', '2025-11-28']
horizons : [1, 2, 4, 8, 13] weeks -> 35 scored points per model


---
## 2. Run every baseline

Hyperparameters live here, next to the results they produce (the repo convention --
see `sp500_forecasting/leaderboard.py`). `num_samples=500` was chosen by measuring
the Monte Carlo wobble: repeat runs vary by ~11 CRPS at 50 samples, ~3 at 500, and
going to 2000 buys nothing (see `cpo.baselines.DEFAULT_NUM_SAMPLES`). `lags=5`
follows the sp500 setup; for `lgbm_diff`, 12 scores the same as 5.


In [2]:
LAGS = 5

PREDICTORS = [
    LastValuePredictor(),
    DartsExponentialSmoothingPredictor(num_samples=DEFAULT_NUM_SAMPLES),
    DartsAutoARIMAPredictor(num_samples=DEFAULT_NUM_SAMPLES),
    DartsKalmanForecasterPredictor(num_samples=DEFAULT_NUM_SAMPLES),
    FixedKalmanPredictor(dim_x=1),
    DartsLightGBMPredictor(
        lags=LAGS,
        lags_past_covariates=None,  # no covariate panel is registered
        num_samples=DEFAULT_NUM_SAMPLES,
        lgbm_kwargs={"num_threads": 1, "n_jobs": 1, "verbosity": -1},
    ),
    DifferencedLightGBMPredictor(lags=LAGS, num_samples=DEFAULT_NUM_SAMPLES),
    DartsLinearRegressionPredictor(lags=LAGS, lags_past_covariates=None, num_samples=DEFAULT_NUM_SAMPLES),
    WeeklyProphetPredictor(),  # yearly seasonality ON, deliberately -- see section 6
    # Bounded-history variant -- see prophet_baseline.py module docstring for
    # why both are kept, and for a third variant (changepoint_range=0.98) that
    # was tried and rejected: it decouples trend recency from seasonality history
    # without truncating anything, fixes the worst origin here (908 -> 279 CRPS),
    # but is worse overall (445 mean CRPS, the worst of all three) because a
    # changepoint that close to the cutoff has no future data to confirm whether
    # it caught a real trend or ordinary noise -- confirmed at 2024-08-30, where
    # it extrapolates a temporary dip as a falling trend for the next 13 weeks.
    # This variant (history_years=4) is NOT a clean fix either: it repairs the
    # trend-boundary problem at the worst origin (908 -> 179 CRPS) but degrades
    # three others, including a new worst case (217 -> 1067 at 2025-02-28) --
    # 4 years is enough training data to bound the changepoint boundary
    # correctly but too few annual cycles (4) to fit yearly seasonality as
    # reliably as the 17-cycle full-history estimate.
    WeeklyProphetPredictor(history_years=4),
    SeasonalNaivePredictor(season_length=52),
]

frames = []
for predictor in PREDICTORS:
    result = run_predictor(predictor, spec, svc)
    frames.append(predictions_frame(result))
    print(f"  {result.predictor_id:22s} mean CRPS {result.mean_score:7.2f}   ({len(result.scores)} points)")

frame = attach_actuals(pd.concat(frames, ignore_index=True), svc)

  last_value_naive       mean CRPS  212.87   (35 points)


  darts_ets              mean CRPS  158.55   (35 points)


  darts_autoarima        mean CRPS  164.41   (35 points)


  darts_kalman           mean CRPS  177.71   (35 points)


  kalman_fixed_dim1      mean CRPS  162.93   (35 points)


...

  darts_linreg           mean CRPS  174.94   (35 points)


11:00:38 - cmdstanpy - INFO - Chain [1] start processing


11:00:38 - cmdstanpy - INFO - Chain [1] done processing


  prophet_weekly         mean CRPS  410.63   (35 points)


  prophet_weekly_4y      mean CRPS  425.53   (35 points)
  seasonal_naive_52      mean CRPS  400.44   (35 points)


---
## 3. Leaderboard

Mean CRPS alone is not enough to rank models -- the views below split it by
horizon (naive wins short range, structure wins long range), by cutoff kind (the
event/quiet design), and by origin (a model that wins on one shock and loses
everywhere else shows up here).


In [3]:
views = summarise(frame)
views["overall"]

,mean_crps
predictor,
darts_ets,158.55
kalman_fixed_dim1,162.93
darts_autoarima,164.41
lgbm_diff,169.70
darts_linreg,174.94
darts_kalman,177.71
last_value_naive,212.87
darts_lightgbm,232.41
seasonal_naive_52,400.44


In [4]:
views["by_horizon"].round(1)

predictor,darts_autoarima,darts_ets,darts_kalman,darts_lightgbm,darts_linreg,kalman_fixed_dim1,last_value_naive,lgbm_diff,prophet_weekly,prophet_weekly_4y,seasonal_naive_52
horizon,,,,,,,,,,,
1,99.4,88.8,91.3,112.5,93.2,92.5,119.4,110.6,432.8,310.2,463.6
2,91.3,67.8,64.0,114.8,89.4,75.4,87.4,107.9,434.9,312.5,426.2
4,110.8,93.6,87.6,94.9,120.8,104.7,108.9,91.7,408.3,309.8,411.0
8,228.2,247.3,313.9,354.2,252.5,249.2,357.8,252.5,318.3,488.3,324.4
13,292.4,295.2,331.7,485.7,318.8,292.8,390.8,285.8,458.8,706.9,377.0


In [5]:
views["by_kind"].round(1)

predictor,darts_autoarima,darts_ets,darts_kalman,darts_lightgbm,darts_linreg,kalman_fixed_dim1,last_value_naive,lgbm_diff,prophet_weekly,prophet_weekly_4y,seasonal_naive_52
kind,,,,,,,,,,,
event,201.3,204.9,241.1,321.5,213.0,202.6,289.4,224.1,498.9,539.4,418.3
quiet,115.2,96.8,93.2,113.6,124.2,110.1,110.9,97.2,292.9,273.7,376.6


In [6]:
per_origin(frame)

predictor,darts_autoarima,darts_ets,darts_kalman,darts_lightgbm,darts_linreg,kalman_fixed_dim1,last_value_naive,lgbm_diff,prophet_weekly,prophet_weekly_4y,seasonal_naive_52
origin_label,,,,,,,,,,,
2024-02-02 event,163.76,139.75,188.32,130.83,189.90,169.10,212.7,191.92,904.22,178.98,237.58
2024-05-03 quiet,127.31,90.19,90.54,92.80,130.02,107.39,93.9,150.59,404.13,263.48,264.88
2024-08-30 event,215.49,257.42,325.29,312.49,256.28,258.14,363.0,264.86,322.18,506.99,441.49
2024-11-29 event,202.64,187.37,189.74,438.85,201.19,176.37,257.3,209.10,552.01,403.32,713.53
2025-02-28 event,223.29,234.89,260.91,403.91,204.62,206.61,324.4,230.44,217.29,1068.44,280.56
2025-06-20 quiet,122.83,119.30,145.05,114.18,138.58,133.04,183.9,70.24,318.13,262.13,242.66
2025-11-28 quiet,95.54,80.92,44.10,133.82,103.99,89.86,54.9,70.78,156.42,295.39,622.38


Skill = fraction of the naive's CRPS removed. **Negative means worse than
assuming nothing ever changes.**


In [7]:
skill_scores(frame)

predictor,darts_autoarima,darts_ets,darts_kalman,darts_lightgbm,darts_linreg,kalman_fixed_dim1,lgbm_diff,prophet_weekly,prophet_weekly_4y,seasonal_naive_52
1,0.168,0.256,0.235,0.058,0.219,0.225,0.074,-2.624,-1.598,-2.882
2,-0.044,0.225,0.268,-0.313,-0.022,0.137,-0.234,-3.974,-2.574,-3.875
4,-0.017,0.141,0.195,0.128,-0.109,0.039,0.158,-2.748,-1.844,-2.773
8,0.362,0.309,0.123,0.010,0.294,0.304,0.294,0.110,-0.365,0.093
13,0.252,0.244,0.151,-0.243,0.184,0.251,0.269,-0.174,-0.809,0.035
all,0.228,0.255,0.165,-0.092,0.178,0.235,0.203,-0.929,-0.999,-0.881


---
## 4. Calibration -- the check CRPS alone cannot do

A `q10`-`q90` band claims to contain the truth 80% of the time. With 7 origins
per horizon the estimate moves in steps of 1/7 ~= 0.14, so read direction, not
digits: well below 0.80 is overconfidence, 1.000 is wider than needed. The naive
is 0.000 by construction (zero-width band); `darts_kalman`'s collapse at 8-13
weeks is the variance bug documented in [`kalman_fixed.py`](kalman_fixed.py).


In [8]:
cov = coverage(frame)
print(f"nominal coverage: {cov.attrs['nominal']:.0%}")
cov

nominal coverage: 80%


predictor,darts_autoarima,darts_ets,darts_kalman,darts_lightgbm,darts_linreg,kalman_fixed_dim1,last_value_naive,lgbm_diff,prophet_weekly,prophet_weekly_4y,seasonal_naive_52
horizon,,,,,,,,,,,
1,0.857,0.857,0.714,0.714,0.857,0.857,0.0,0.286,0.571,0.714,0.714
2,0.857,1.000,1.000,0.714,1.000,1.000,0.0,0.429,0.571,0.714,0.714
4,1.000,1.000,0.714,0.714,1.000,1.000,0.0,0.857,0.714,0.571,0.714
8,1.000,0.429,0.286,0.571,1.000,0.857,0.0,0.429,0.857,0.571,1.000
13,0.714,0.714,0.286,0.429,0.857,0.714,0.0,0.714,0.571,0.143,0.857


---
## 5. The pictures

CRPS by horizon first: the crossover between the naive and everything else is
the story of this series -- a random walk is near-unbeatable at 1-2 weeks, and
honest uncertainty pays from 4 weeks out.


In [9]:
core = frame[frame.predictor.isin(["last_value_naive", "darts_ets", "darts_autoarima", "kalman_fixed_dim1"])]
plot_crps_by_horizon(core)

The fan grid is the headline figure: one predictor across all seven cutoffs,
events on top, quiets below, one shared y-scale. Read it as *"was the model
calibrated, and where did it earn its score"* -- e.g. 2024-08-30 is the
expensive panel because the realised rally rides the upper edge of the band.


In [10]:
history = svc.get_series(MPOB_WEEKLY_SERIES_ID, as_of=naive_utc_now())
plot_fan_grid(frame, history, predictor="darts_ets")

The same grid for the naive is the contrast that explains the whole leaderboard:
an identical point forecast with no band at all.


In [11]:
plot_fan_grid(frame, history, predictor="last_value_naive")

The fan grids above are one model at a time -- good for a deep dive, hard to use for
direct comparison since each is its own figure. This overlays every predictor's median
forecast on the same axes: click a name in the legend to isolate it, click again to bring
it back, and compare any subset directly against `actual` and against each other.


In [12]:
plot_predictor_comparison(frame, history)

---
## 6. Monte Carlo noise -- how big must a gap be to mean anything?

The sampled predictors rescore differently every run. Any leaderboard gap inside
this spread is sampling noise, not evidence.


In [13]:
noise = mc_noise(
    lambda: DartsExponentialSmoothingPredictor(num_samples=DEFAULT_NUM_SAMPLES), runs=5, spec=spec, data_service=svc
)
print(f"darts_ets over {int(noise['runs'])} runs: mean {noise['mean']:.1f}, spread {noise['spread']:.1f} CRPS")
print("-> ETS vs AutoARIMA vs kalman_fixed (~157-164) is a statistical tie on 35 points.")

darts_ets over 5 runs: mean 157.0, spread 3.6 CRPS
-> ETS vs AutoARIMA vs kalman_fixed (~157-164) is a statistical tie on 35 points.


---
## 7. Foundation models -- zero-shot, leakage-gated

Chronos (Amazon) and TimesFM (Google) are pretrained on public time series and never see this series during training -- a genuinely different kind of predictor from everything above, all of which fit fresh at each cutoff. The risk that comes with that: neither vendor publishes an exact pretraining data cutoff, so a checkpoint could in principle have already seen the real prices we are asking it to forecast.

The fix: a checkpoint's public release date is always *after* its training data was collected (training, evaluation, and release all take real time), so a cutoff is safe for a given checkpoint only if the checkpoint was already released on or before that cutoff. `cpo.chronos_baseline` and `cpo.timesfm_baseline` enforce this in code -- `predict()` returns `[]` for any cutoff before the checkpoint's release date, so an unsafe pairing produces no row at all rather than a silently leaked one. Full reasoning, the per-checkpoint release-date table (sourced from the Hugging Face API, not the announcement date), and two upstream library defects found and fixed while building this -- Chronos-Bolt silently clipping its 0.05/0.95 tails, TimesFM's own `fix_quantile_crossing` flag missing one of nine quantile levels -- are documented in both modules' docstrings.

Coverage is uneven and small on purpose -- these are the checkpoints that were actually safe to test at each cutoff, nothing was excluded to flatter the numbers:

| checkpoint | released | safe cutoffs | points |
|---|---|---|---|
| `chronos_t5_small` | 2024-02-21 | 6 of 7 | 30 |
| `chronos_bolt_base` | 2024-11-25 | 4 of 7 | 20 |
| `chronos_2` | 2025-10-30 | 1 of 7 (2025-11-28) | 5 |
| `timesfm_2_5_200m_pytorch` | 2025-09-02 | 1 of 7 (2025-11-28) | 5 |

**These are kept out of the leaderboard in section 3 on purpose.** `chronos_2` and `timesfm_2_5`'s 5 points come from a single realised price path -- one origin, five correlated horizons -- which is one data point, not five, and nowhere near enough to say anything about skill. `chronos_t5_small` and `chronos_bolt_base` have more coverage but are still missing the hardest early cutoffs, so a bare mean-CRPS column next to the fully-scored 35-point models above would misrepresent them as tested the same way. The same-origin table below is the one honest comparison the data actually supports.


In [14]:
import gc

from cpo.chronos_baseline import WeeklyChronosPredictor
from cpo.timesfm_baseline import WeeklyTimesFMPredictor


# Constructed one at a time (not kept in a list) so each checkpoint's cached
# weights are garbage-collected before the next one loads -- see the module
# docstrings for the SIGSEGV this avoids when running alongside darts_lightgbm
# in the same kernel.
FOUNDATION_MODEL_SPECS = [
    (
        WeeklyChronosPredictor,
        {
            "checkpoint": "amazon/chronos-t5-small",
            "revision": "a971ba21945c4f1796b17a91fe69214b5f4ad472",
            "release_date": "2024-02-21",
        },
    ),
    (
        WeeklyChronosPredictor,
        {
            "checkpoint": "amazon/chronos-bolt-base",
            "revision": "5d9f166d69f47aef3401367a7b842e78fe97b121",
            "release_date": "2024-11-25",
        },
    ),
    (
        WeeklyChronosPredictor,
        {
            "checkpoint": "amazon/chronos-2",
            "revision": "29ec3766d36d6f73f0696f85560a422f50e8498c",
            "release_date": "2025-10-30",
        },
    ),
    (
        WeeklyTimesFMPredictor,
        {
            "checkpoint": "google/timesfm-2.5-200m-pytorch",
            "revision": "1d952420fba87f3c6dee4f240de0f1a0fbc790e3",
            "release_date": "2025-09-02",
        },
    ),
]

fm_frames = []
for predictor_cls, kwargs in FOUNDATION_MODEL_SPECS:
    predictor = predictor_cls(**kwargs)
    result = run_predictor(predictor, spec, svc)
    pf = predictions_frame(result)
    fm_frames.append(pf)
    print(
        f"  {result.predictor_id:26s} mean CRPS {result.mean_score:7.2f}   ({len(result.scores)} points, {pf['origin'].nunique()} cutoffs)"
    )
    del predictor, result
    gc.collect()

fm_frame = attach_actuals(pd.concat(fm_frames, ignore_index=True), svc)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

  chronos_t5_small           mean CRPS  173.91   (30 points, 6 cutoffs)


Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

  chronos_bolt_base          mean CRPS  152.10   (20 points, 4 cutoffs)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

  chronos_2                  mean CRPS   70.83   (5 points, 1 cutoffs)


  timesfm_2_5_200m_pytorch   mean CRPS   95.98   (5 points, 1 cutoffs)


Each row below is scored on its own point count, not the shared 35 -- read `points`/`cutoffs` before comparing mean CRPS across rows, and see section 7's intro for why `chronos_2` and `timesfm_2_5` are one data point each, not five.

In [15]:
fm_summary = (
    fm_frame.groupby("predictor")
    .agg(mean_crps=("crps", "mean"), points=("crps", "size"), cutoffs=("origin", "nunique"))
    .round(1)
)
fm_summary

,mean_crps,points,cutoffs
predictor,,,
chronos_2,70.8,5,1
chronos_bolt_base,152.1,20,4
chronos_t5_small,173.9,30,6
timesfm_2_5_200m_pytorch,96.0,5,1


**The one honest comparison:** all fifteen predictors scored on the single cutoff every foundation model here has in common, `2025-11-28`. This is a real apples-to-apples ranking -- unlike the table above, every row is five points from the same origin.

In [16]:
shared_origin = pd.Timestamp("2025-11-28")
same_origin = pd.concat([frame, fm_frame], ignore_index=True)
same_origin[same_origin["origin"] == shared_origin].groupby("predictor")["crps"].mean().sort_values().round(1)

predictor
darts_kalman                 44.1
last_value_naive             54.9
lgbm_diff                    70.8
chronos_2                    70.8
chronos_bolt_base            74.1
darts_ets                    80.9
kalman_fixed_dim1            89.9
darts_autoarima              95.5
timesfm_2_5_200m_pytorch     96.0
darts_linreg                104.0
chronos_t5_small            119.6
darts_lightgbm              133.8
prophet_weekly              156.4
prophet_weekly_4y           295.4
seasonal_naive_52           622.4
Name: crps, dtype: float64

---
## 8. What this establishes, and what it does not

**Established:**

- **Every working model converges on the random walk.** ETS fits alpha = 1 at all
  seven origins, AutoARIMA selects d=1 everywhere, N4SID identifies |eig| = 0.999,
  linreg's lag coefficients sum to ~1. Their entire advantage over the naive
  (~26% CRPS) is calibrated uncertainty around the same point forecast.
- **The calendar-structure hypothesis fails.** Prophet (trend + yearly cycle) and
  the 52-week seasonal naive lose to the floor by ~2x: palm oil in this sample has
  no annual cycle worth modelling. Prophet also has no autoregressive term, so its
  1-week forecast sits ~190 RM from the last price where ARIMA sits ~40 RM.
  (Tuning its changepoints improves 411 -> ~227 but cannot fix the anchoring;
  the committed config keeps the hypothesis-test defaults.)
- **Two upstream defects were found, diagnosed, and fixed locally** --
  `darts_kalman`'s frozen multi-step variance ([`kalman_fixed.py`](kalman_fixed.py))
  and LightGBM's level-feature regime matching ([`lgbm_differenced.py`](lgbm_differenced.py)).
  Report neither package model's score without the caveat.
- **The bar for the agent:** beat ~157 CRPS not by better point forecasts (four
  estimators say there is nothing to find in the price series alone) but by
  anticipating event windows from news and staying calibrated on quiet ones.

**Limitations:**

- **35 points cannot separate close models.** ETS / AutoARIMA / kalman_fixed sit
  within ~6 CRPS of each other with ~3 CRPS of MC wobble -- a tie. The dense
  weekly backtest (`cpo_backtest.yaml`, 52 origins) is the instrument for picking
  a single champion; these seven origins are the narrative set.
- **Coverage on 7 origins is directional only** (steps of 1/7).
- **Event cutoffs were chosen with hindsight** -- valid for the controlled
  comparison, not a live forecasting record.
- **No 2022-scale shock is in the window.** The largest move any model faces here
  is ~7.7%; behaviour under a 30% shock is untested.

**Next:** the news-reading agent on the same spec, compared against `darts_ets`
on these same tables.
